## 중간값 필터를 이용하여 배경 추정하기

### 기본 라이브러리 불러오기

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

### median filter user defined function

In [2]:
def median_filter(buffer):
    new_b = np.median(buffer)
    return new_b

### User parameter 정의하기

In [3]:
delay = 30 # 영상 출력 속도 제어
threshold = 100
n = 121 # 버퍼의 크기

### "Video_003.avi"를 읽어오기

In [4]:
capture = cv2.VideoCapture("images/Video_003.avi")

# 잘 읽어 왔는지 검사하는 코드
if capture.isOpened() == False: 
    raise Exception("Error")

# 재생할 파일의 너비와 높이
w = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

### 비디오 분할하기

In [7]:
# 프레임 번호 초기화(t) + 버퍼 초기화(buffer)
t = 1
buffer = np.zeros((h,w,n))

while True:
    # 컬러 프레임 읽어오기(f_color)
    ret, f_color = capture.read()  # 카메라 영상 받기

    if not ret: # 받아들인 프레임이 존재하는가? (T/F)
        break
    else:
        # 흑백 영상으로 변환하기(f)
        f = cv2.cvtColor(f_color, cv2.COLOR_BGR2GRAY)

        if t <= n: # t -> 0~2(n장)
                    # 버퍼에 쌓기
                    buffer [:,:,t-1] = f
        else: # 두번째 프레임부터~            
            # median filter로부터 배경 추정하기(b)
            b = median_filter(buffer)

            # 절대차(absolute difference) 구하기(diff)
            diff = cv2.absdiff(f.astype("float"),b.astype("float"))

            # 임계값을 활용하여 결과 영상 생성하기(g)
            g = np.where(diff>threshold,255,0)

            # 시간 지연 및 종료 시킬지 여부
            if cv2.waitKey(delay) >= 0:
                break

            # 결과 확인하기
            res = cv2.vconcat([cv2.hconcat([f.astype("uint8"), b.astype("uint8")]), 
                               cv2.hconcat([diff.astype("uint8"), g.astype("uint8")])])
            cv2.imshow("result image", res)

            # 버퍼 업데이트 하기
            buffer [:,:,0:n-1]= buffer [:,:,1:]
            buffer [:,:,n-1]= f
            
    # 프레임 번호 업데이트(t)
    t += 1

capture.release()
cv2.destroyAllWindows()

error: OpenCV(5.0.0) /io/opencv/modules/highgui/src/window.cpp:1262: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'waitKeyImpl'
